# dnabert2-clinvar-kfp-sequence-classify

[![Open in JupyterLab](https://img.shields.io/badge/Open%20in-JupyterLab-F37626?logo=jupyter&logoColor=white)](http://localhost:8888/lab/tree/git-miramar-labs-org/projects/dnabert2-clinvar-kfp-sequence-classify/notebook.ipynb)  [![last run](https://img.shields.io/badge/last%20run-pending-lightgrey)](runs/RUNS.md)

| | |
|---|---|
| **Type** | KFP v2 sequence-encoder classification pipeline |
| **Model** | [zhihan1996/DNABERT-2-117M](https://huggingface.co/zhihan1996/DNABERT-2-117M) |
| **Dataset** | [wanglab/variant_effect_coding](https://huggingface.co/datasets/wanglab/variant_effect_coding) |

DNABERT-2 sequence-encoder classification pipeline for ClinVar variant pathogenicity prediction

---

## What this pipeline does

This is a **transfer learning** pipeline built in two phases. Understanding both phases
is essential because they answer different questions.

**Phase 1 — Pre-training (already done by the model authors, not by this pipeline)**

The base model (e.g. DNABERT-2) was trained on billions of raw sequence letters using
a *masked prediction* task — randomly hide 15% of the input and ask the model to fill
them in. No labels. No task. Just "learn the structure of this sequence space."

At the end of Phase 1, the model outputs a dense vector (the `[CLS]` token embedding)
that compresses everything it has learned about a sequence into 768 numbers. This vector
has **no concept of "pathogenic" or "benign"** — it only knows the sequence distribution.

We download these weights from HuggingFace and use them as the starting point.

**Phase 2 — Classification fine-tuning (what this pipeline does)**

We attach a small **linear classification head** on top of the `[CLS]` token and train
the full system on labeled examples. After fine-tuning, the head has learned what the
"positive class" embedding looks like in the pre-trained representation space.

```
sequence → [pre-trained encoder] → [CLS] vector → [linear head] → class probability
```

**Why the baseline is ~50% (not the model's knowledge):**

The baseline evaluation runs *before* fine-tuning. The classification head is
randomly initialized — it has never seen a label. Expected baseline: accuracy ≈ 50%,
AUC-ROC ≈ 0.50 for binary classification. This is correct and expected.

Compare to `kfp-ft-eval`: there, the LLM *already knows* the task domain (e.g.
MedGemma has medical knowledge → 75% baseline). Here, the head starts random.
The large delta after fine-tuning is the signal.

**Pipeline DAG:**
```
download_model ──► prepare_dataset ──► baseline_eval ──► fine_tune ──► post_finetune_eval ──► deployment_gate
```

***

## Step Development

Each `@dsl.component` cell below is tagged `kfp_step`. The pipeline cell is tagged
`kfp_pipeline`. Only tagged cells are included in `pipeline.py`.

**Edit → build → compile cycle:**
```bash
python3 scripts/build_pipeline.py
python3 -c "from kfp import compiler; from pipeline import pipeline; \
    compiler.Compiler().compile(pipeline, '/tmp/p.yaml'); print('OK')"
```

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact
from typing import NamedTuple

***

## `download_model`

Snapshot-download the base encoder weights from HuggingFace Hub.
No GPU needed. Fully implemented — no user code required.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["huggingface_hub>=0.23"],
)
def download_model(
    model_id: str,
    base_model: Output[Model],
) -> None:
    import os
    from huggingface_hub import snapshot_download

    token = os.environ.get("HF_TOKEN")
    snapshot_download(
        repo_id=model_id,
        local_dir=base_model.path,
        token=token,
        ignore_patterns=["*.msgpack", "flax_model*", "tf_model*", "rust_model*"],
    )
    print(f"Downloaded {model_id} to {base_model.path}")

***

## `prepare_dataset`

Load sequences via `processors.py`, then split into train / val / test.

**Split strategies:**
- `chromosome`: hold out specific chromosomes as val and test sets. Prevents eval leakage
  from nearby correlated sequences (variants on the same chromosome share haplotype patterns).
  Requires `chromosome` field in processor output rows.
- `random`: shuffle by seed and split by fraction.

Each row written to the split files:
```json
{"sequence": "ATCG...", "label": 1, "source": "variant_effect_coding", "chromosome": "chr3"}
```

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["datasets>=2.0", "huggingface_hub>=0.23", "pyyaml"],
)
def prepare_dataset(
    config_yaml: str,
    train_data: Output[Dataset],
    val_data: Output[Dataset],
    test_data: Output[Dataset],
) -> None:
    import json
    import os
    import random
    import yaml

    cfg = yaml.safe_load(config_yaml)
    dataset_cfg = cfg["dataset"]
    training_cfg = cfg["training"]

    split_strategy = dataset_cfg.get("split_strategy", "random")
    val_size = training_cfg.get("val_size", 0.1)
    test_size = training_cfg.get("test_size", 0.1)
    seed = training_cfg.get("shuffle_seed", 42)

    dataset_id = dataset_cfg["id"]
    loader_key = dataset_cfg.get("loader_key") or dataset_id.split("/")[-1]

    # <<< PROCESSORS_INJECT >>>

    if loader_key not in LOADERS:  # noqa: F821
        raise ValueError(
            f"No loader for '{loader_key}'. "
            f"Available: {list(LOADERS.keys())}. "
            f"Add it to processors.py or set dataset.loader_key in config.yaml."
        )

    rows = list(LOADERS[loader_key]())  # noqa: F821
    print(f"Loaded {len(rows)} rows from {dataset_id}")

    if split_strategy == "chromosome":
        test_chrom = dataset_cfg.get("test_chromosome", "chr22")
        val_chrom = dataset_cfg.get("val_chromosome", "chr21")
        train_rows = [r for r in rows if r.get("chromosome") not in (test_chrom, val_chrom)]
        val_rows   = [r for r in rows if r.get("chromosome") == val_chrom]
        test_rows  = [r for r in rows if r.get("chromosome") == test_chrom]
        print(f"Chromosome split: train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}")
        print(f"  test: {test_chrom}  val: {val_chrom}")
        if len(val_rows) == 0 or len(test_rows) == 0:
            raise ValueError(
                f"Chromosome split produced empty val or test set. "
                f"Check that processor outputs 'chromosome' field and that "
                f"test_chromosome='{test_chrom}' / val_chromosome='{val_chrom}' exist in the data."
            )
    else:
        rng = random.Random(seed)
        rng.shuffle(rows)
        n = len(rows)
        n_test = max(1, int(n * test_size))
        n_val  = max(1, int(n * val_size))
        test_rows  = rows[:n_test]
        val_rows   = rows[n_test:n_test + n_val]
        train_rows = rows[n_test + n_val:]
        print(f"Random split (seed={seed}): train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}")

    for path, data in [
        (train_data.path, train_rows),
        (val_data.path, val_rows),
        (test_data.path, test_rows),
    ]:
        os.makedirs(path, exist_ok=True)
        with open(os.path.join(path, "data.jsonl"), "w") as f:
            for row in data:
                f.write(json.dumps(row) + "\n")

    print(f"Splits written: train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}")

***

## `baseline_eval`

Load the pre-trained encoder with a **randomly initialized** classification head.
Evaluate on the validation set.

Expected results: accuracy ≈ 50%, AUC-ROC ≈ 0.50.

This is the correct baseline — the head has never seen a label. The large delta
after fine-tuning is the meaningful signal.

`# <<< UTILS_INJECT >>>` brings `compute_auc` from `utils.py` into scope.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=["transformers>=4.40", "scikit-learn", "mlflow", "pyyaml"],
)
def baseline_eval(
    base_model: Input[Model],
    val_data: Input[Dataset],
    config_yaml: str,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
) -> NamedTuple("Outputs", baseline_accuracy=float, baseline_auc=float):
    import json
    import os
    import random
    import yaml
    import torch
    import mlflow

    cfg = yaml.safe_load(config_yaml)
    num_labels = cfg["model"].get("num_labels", 2)
    max_length = cfg["training"].get("max_length", 512)
    sample_size = cfg["eval"].get("sample_size", 500)

    # <<< UTILS_INJECT >>>

    val_rows = []
    with open(os.path.join(val_data.path, "data.jsonl")) as f:
        for line in f:
            row = json.loads(line.strip())
            if row:
                val_rows.append(row)

    if len(val_rows) > sample_size:
        random.seed(42)
        val_rows = random.sample(val_rows, sample_size)
    print(f"Evaluating on {len(val_rows)} val samples (baseline — random head)")

    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    tokenizer = AutoTokenizer.from_pretrained(base_model.path, trust_remote_code=True)
    from transformers import AutoConfig as _AutoConfig
    _config = _AutoConfig.from_pretrained(base_model.path, trust_remote_code=True)
    if not hasattr(_config, 'pad_token_id') or _config.pad_token_id is None:
        _config.pad_token_id = 3
    _config.num_labels = num_labels
    import os as _os
    model = AutoModelForSequenceClassification.from_config(_config, trust_remote_code=True)
    import sys as _sys
    for _mn, _mm in list(_sys.modules.items()):
        if 'bert_layers' in _mn and hasattr(_mm, 'flash_attn_qkvpacked_func'):
            _mm.flash_attn_qkvpacked_func = None  # bundled flash_attn_triton uses removed Triton trans_b API
    _sf = _os.path.join(base_model.path, 'model.safetensors')
    _pb = _os.path.join(base_model.path, 'pytorch_model.bin')
    if _os.path.exists(_sf):
        from safetensors.torch import load_file as _lf
        _psd = _lf(_sf, device='cpu')
    elif _os.path.exists(_pb):
        _psd = torch.load(_pb, map_location='cpu', weights_only=True)
    else:
        _psd = {}
    if _psd:
        _msd = model.state_dict()
        model.load_state_dict({k: v for k, v in _psd.items() if k in _msd and _msd[k].shape == v.shape}, strict=False)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    print(f"Model on {device} — randomly initialized head (num_labels={num_labels})")
    print(f"Expected: accuracy ≈ {100 // num_labels}%, AUC ≈ 0.50")

    preds, labels_list, probs_positive = [], [], []

    with torch.no_grad():
        for row in val_rows:
            enc = tokenizer(
                row["sequence"],
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                padding="max_length",
            ).to(device)
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).squeeze(0)
            preds.append(int(torch.argmax(probs).item()))
            labels_list.append(int(row["label"]))
            probs_positive.append(float(probs[1].item()) if num_labels == 2 else float(probs.max().item()))

    accuracy = sum(p == l for p, l in zip(preds, labels_list)) / len(labels_list)
    auc = compute_auc(labels_list, probs_positive)  # noqa: F821

    print(f"Baseline accuracy: {accuracy:.4f}")
    print(f"Baseline AUC-ROC:  {auc:.4f}")

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/baseline_eval", nested=True):
        mlflow.log_metric("baseline_accuracy", accuracy)
        mlflow.log_metric("baseline_auc", auc)

    metrics.log_metric("baseline_accuracy", accuracy)
    metrics.log_metric("baseline_auc", auc)

    from collections import namedtuple
    Outputs = namedtuple("Outputs", ["baseline_accuracy", "baseline_auc"])
    return Outputs(accuracy, auc)

***

## `fine_tune`

Train the encoder + classification head using HuggingFace `Trainer`.

Default: full fine-tune (`use_lora: false`). Recommended for small encoders (< 500M params).
DNABERT-2 is 117M params — full fine-tune fits in < 4 GB GPU memory.

Set `training.use_lora: true` in `config.yaml` for larger models (1B+) to use PEFT LoRA.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=["transformers>=4.40", "datasets>=2.0", "mlflow", "pyyaml", "peft"],
)
def fine_tune(
    base_model: Input[Model],
    train_data: Input[Dataset],
    val_data: Input[Dataset],
    config_yaml: str,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    ft_model: Output[Model],
    metrics: Output[Metrics],
) -> None:
    import json
    import os
    import yaml
    import torch
    import mlflow

    cfg = yaml.safe_load(config_yaml)
    model_cfg    = cfg["model"]
    training_cfg = cfg["training"]

    num_labels   = int(model_cfg.get("num_labels", 2))
    max_length   = int(training_cfg.get("max_length", 512))
    num_epochs   = int(training_cfg.get("num_epochs", 3))
    learning_rate = float(training_cfg.get("learning_rate", 2e-5))
    batch_size   = int(training_cfg.get("batch_size", 32))
    grad_accum   = int(training_cfg.get("gradient_accumulation_steps", 1))
    use_lora     = bool(training_cfg.get("use_lora", False))
    weight_decay = float(training_cfg.get("weight_decay", 0.01))

    def load_jsonl(path):
        rows = []
        with open(os.path.join(path, "data.jsonl")) as f:
            for line in f:
                row = json.loads(line.strip())
                if row:
                    rows.append(row)
        return rows

    train_rows = load_jsonl(train_data.path)
    val_rows   = load_jsonl(val_data.path)
    print(f"Train: {len(train_rows)}, Val: {len(val_rows)}")

    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        Trainer,
        TrainingArguments,
    )
    from torch.utils.data import Dataset as TorchDataset

    tokenizer = AutoTokenizer.from_pretrained(base_model.path, trust_remote_code=True)

    from transformers import AutoConfig as _AutoConfig
    _config = _AutoConfig.from_pretrained(base_model.path, trust_remote_code=True)
    if not hasattr(_config, 'pad_token_id') or _config.pad_token_id is None:
        _config.pad_token_id = 3
    _config.num_labels = num_labels
    import os as _os
    model = AutoModelForSequenceClassification.from_config(_config, trust_remote_code=True)
    import sys as _sys
    for _mn, _mm in list(_sys.modules.items()):
        if 'bert_layers' in _mn and hasattr(_mm, 'flash_attn_qkvpacked_func'):
            _mm.flash_attn_qkvpacked_func = None  # bundled flash_attn_triton uses removed Triton trans_b API
    _sf = _os.path.join(base_model.path, 'model.safetensors')
    _pb = _os.path.join(base_model.path, 'pytorch_model.bin')
    if _os.path.exists(_sf):
        from safetensors.torch import load_file as _lf
        _psd = _lf(_sf, device='cpu')
    elif _os.path.exists(_pb):
        _psd = torch.load(_pb, map_location='cpu', weights_only=True)
    else:
        _psd = {}
    if _psd:
        _msd = model.state_dict()
        model.load_state_dict({k: v for k, v in _psd.items() if k in _msd and _msd[k].shape == v.shape}, strict=False)

    if use_lora:
        from peft import LoraConfig, get_peft_model, TaskType
        lora_cfg = cfg.get("lora", {})
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=int(lora_cfg.get("r", 8)),
            lora_alpha=int(lora_cfg.get("alpha", 16)),
            lora_dropout=float(lora_cfg.get("dropout", 0.1)),
            target_modules=lora_cfg.get("target_modules") or None,
            bias="none",
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
    else:
        n_params = sum(p.numel() for p in model.parameters())
        print(f"Full fine-tune: {n_params / 1e6:.1f}M parameters")

    class SeqDataset(TorchDataset):
        def __init__(self, rows):
            seqs = [r["sequence"] for r in rows]
            self.encodings = tokenizer(
                seqs,
                truncation=True,
                max_length=max_length,
                padding="max_length",
            )
            self.labels = [int(r["label"]) for r in rows]

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return {
                "input_ids":      torch.tensor(self.encodings["input_ids"][idx],      dtype=torch.long),
                "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
                "labels":         torch.tensor(self.labels[idx],                      dtype=torch.long),
            }

    train_dataset = SeqDataset(train_rows)
    val_dataset   = SeqDataset(val_rows)
    print(f"Tokenized: train={len(train_dataset)}, val={len(val_dataset)}")

    os.makedirs(ft_model.path, exist_ok=True)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)

    training_args = TrainingArguments(
        output_dir=ft_model.path,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        gradient_accumulation_steps=grad_accum,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_steps=10,
        bf16=True,
        report_to=["mlflow"],
        run_name=f"{run_id}/fine_tune",
        dataloader_num_workers=0,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    with mlflow.start_run(run_name=f"{run_id}/fine_tune", nested=True):
        mlflow.log_params({
            "model_id":     model_cfg.get("id", ""),
            "num_labels":   num_labels,
            "num_epochs":   num_epochs,
            "batch_size":   batch_size,
            "learning_rate": learning_rate,
            "max_length":   max_length,
            "use_lora":     use_lora,
        })
        result = trainer.train()
        train_loss = result.training_loss
        mlflow.log_metric("train_loss", train_loss)

    model.save_pretrained(ft_model.path)
    tokenizer.save_pretrained(ft_model.path)

    print(f"Fine-tune complete — loss: {train_loss:.4f}")
    print(f"Model saved to {ft_model.path}")
    metrics.log_metric("train_loss", train_loss)

***

## `post_finetune_eval`

Load the fine-tuned model from `ft_model.path` and evaluate on the **held-out test set**.

Uses the test set (not val) for an unbiased estimate of generalization.
The deployment gate compares `postft_auc` against the `gate_auc_threshold` in `config.yaml`.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=["transformers>=4.40", "scikit-learn", "mlflow", "pyyaml", "peft"],
)
def post_finetune_eval(
    ft_model: Input[Model],
    test_data: Input[Dataset],
    config_yaml: str,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
) -> NamedTuple("Outputs", postft_accuracy=float, postft_auc=float):
    import json
    import os
    import yaml
    import torch
    import mlflow

    cfg = yaml.safe_load(config_yaml)
    num_labels = cfg["model"].get("num_labels", 2)
    max_length = cfg["training"].get("max_length", 512)

    # <<< UTILS_INJECT >>>

    test_rows = []
    with open(os.path.join(test_data.path, "data.jsonl")) as f:
        for line in f:
            row = json.loads(line.strip())
            if row:
                test_rows.append(row)
    print(f"Evaluating on {len(test_rows)} test samples (post-FT)")

    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    tokenizer = AutoTokenizer.from_pretrained(ft_model.path, trust_remote_code=True)
    from transformers import AutoConfig as _AutoConfig
    _config = _AutoConfig.from_pretrained(ft_model.path, trust_remote_code=True)
    if not hasattr(_config, 'pad_token_id') or _config.pad_token_id is None:
        _config.pad_token_id = 3
    _config.num_labels = num_labels
    import os as _os
    model = AutoModelForSequenceClassification.from_config(_config, trust_remote_code=True)
    import sys as _sys
    for _mn, _mm in list(_sys.modules.items()):
        if 'bert_layers' in _mn and hasattr(_mm, 'flash_attn_qkvpacked_func'):
            _mm.flash_attn_qkvpacked_func = None  # bundled flash_attn_triton uses removed Triton trans_b API
    _sf = _os.path.join(ft_model.path, 'model.safetensors')
    _pb = _os.path.join(ft_model.path, 'pytorch_model.bin')
    if _os.path.exists(_sf):
        from safetensors.torch import load_file as _lf
        _psd = _lf(_sf, device='cpu')
    elif _os.path.exists(_pb):
        _psd = torch.load(_pb, map_location='cpu', weights_only=True)
    else:
        _psd = {}
    if _psd:
        _msd = model.state_dict()
        model.load_state_dict({k: v for k, v in _psd.items() if k in _msd and _msd[k].shape == v.shape}, strict=False)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    print(f"Fine-tuned model loaded on {device}")

    preds, labels_list, probs_positive = [], [], []

    with torch.no_grad():
        for row in test_rows:
            enc = tokenizer(
                row["sequence"],
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                padding="max_length",
            ).to(device)
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).squeeze(0)
            preds.append(int(torch.argmax(probs).item()))
            labels_list.append(int(row["label"]))
            probs_positive.append(float(probs[1].item()) if num_labels == 2 else float(probs.max().item()))

    accuracy = sum(p == l for p, l in zip(preds, labels_list)) / len(labels_list)
    auc = compute_auc(labels_list, probs_positive)  # noqa: F821

    print(f"Post-FT accuracy: {accuracy:.4f}")
    print(f"Post-FT AUC-ROC:  {auc:.4f}")

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/post_finetune_eval", nested=True):
        mlflow.log_metric("postft_accuracy", accuracy)
        mlflow.log_metric("postft_auc", auc)

    metrics.log_metric("postft_accuracy", accuracy)
    metrics.log_metric("postft_auc", auc)

    from collections import namedtuple
    Outputs = namedtuple("Outputs", ["postft_accuracy", "postft_auc"])
    return Outputs(accuracy, auc)

***

## `deployment_gate`

Gate on two conditions (from `config.yaml`):
- `accuracy_delta >= eval.gate_accuracy_delta` (e.g. post-FT − baseline ≥ −0.02)
- `postft_auc >= eval.gate_auc_threshold` (e.g. AUC ≥ 0.70)

Raises `RuntimeError` on failure, which marks the KFP run as FAILED.
Writes `gate_result.json` on success.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["pyyaml", "mlflow"],
)
def deployment_gate(
    baseline_accuracy: float,
    baseline_auc: float,
    postft_accuracy: float,
    postft_auc: float,
    config_yaml: str,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    gate_result: Output[Artifact],
) -> None:
    import json
    import os
    import yaml
    import mlflow

    cfg = yaml.safe_load(config_yaml)
    eval_cfg = cfg.get("eval", {})
    delta_threshold = float(eval_cfg.get("gate_accuracy_delta", -0.02))
    auc_threshold   = float(eval_cfg.get("gate_auc_threshold", 0.70))

    accuracy_delta = postft_accuracy - baseline_accuracy

    checks = {
        "accuracy_delta": {
            "value":     accuracy_delta,
            "threshold": delta_threshold,
            "passed":    accuracy_delta >= delta_threshold,
        },
        "postft_auc": {
            "value":     postft_auc,
            "threshold": auc_threshold,
            "passed":    postft_auc >= auc_threshold,
        },
    }

    gate_passed = all(c["passed"] for c in checks.values())

    print(f"Baseline accuracy:  {baseline_accuracy:.4f}")
    print(f"Post-FT accuracy:   {postft_accuracy:.4f}")
    delta_mark = "\u2713" if checks["accuracy_delta"]["passed"] else "\u2717"
    print(f"Accuracy delta:     {accuracy_delta:+.4f}  (threshold: {delta_threshold:+.4f}) {delta_mark}")
    print(f"Baseline AUC-ROC:   {baseline_auc:.4f}")
    auc_mark = "\u2713" if checks["postft_auc"]["passed"] else "\u2717"
    print(f"Post-FT AUC-ROC:    {postft_auc:.4f}  (threshold: {auc_threshold:.4f}) {auc_mark}")
    print(f"\nGate: {'PASS' if gate_passed else 'FAIL'}")

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/deployment_gate", nested=True):
        mlflow.log_metric("gate_passed", 1.0 if gate_passed else 0.0)
        mlflow.log_metric("accuracy_delta", accuracy_delta)
        mlflow.log_metric("postft_auc", postft_auc)

    result = {
        "gate_passed":       gate_passed,
        "baseline_accuracy": baseline_accuracy,
        "postft_accuracy":   postft_accuracy,
        "accuracy_delta":    accuracy_delta,
        "baseline_auc":      baseline_auc,
        "postft_auc":        postft_auc,
        "checks":            checks,
    }

    os.makedirs(gate_result.path, exist_ok=True)
    with open(os.path.join(gate_result.path, "gate_result.json"), "w") as f:
        json.dump(result, f, indent=2)

    if not gate_passed:
        failed = [k for k, v in checks.items() if not v["passed"]]
        raise RuntimeError(f"Deployment gate FAILED: {', '.join(failed)}")

***

## Pipeline

Wire the components. The pipeline reads `config.yaml` at compile time to populate
default parameter values.

In [ ]:
import os as _os, yaml as _yaml

_cfg = _yaml.safe_load(open("config.yaml").read()) if _os.path.exists("config.yaml") else {}
_model_id   = _cfg.get("model", {}).get("id", "{{HF_MODEL_ID}}")
_dataset_id = _cfg.get("dataset", {}).get("id", "{{HF_DATASET_ID}}")
_config_yaml_default = open("config.yaml").read() if _os.path.exists("config.yaml") else ""


@dsl.pipeline(name="dnabert2-clinvar-kfp-sequence-classify")
def pipeline(
    model_id: str = _model_id,
    dataset_id: str = _dataset_id,
    config_yaml: str = "",
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
    mlflow_experiment_name: str = "dnabert2-clinvar-kfp-sequence-classify",
):
    from kfp import kubernetes as k8s_ext

    _SECRET = "mlabs-api-keys"
    _hf_map = {"HF_TOKEN": "HF_TOKEN"}

    # ── download model ───────────────────────────────────────────────────────
    dl = download_model(model_id=model_id)
    dl.set_memory_limit("16G")
    k8s_ext.use_secret_as_env(dl, secret_name=_SECRET, secret_key_to_env=_hf_map)

    # ── prepare dataset ─────────────────────────────────────────────────────
    prep = prepare_dataset(config_yaml=config_yaml)
    k8s_ext.use_secret_as_env(prep, secret_name=_SECRET, secret_key_to_env=_hf_map)

    # ── baseline eval ───────────────────────────────────────────────────────────
    baseline = baseline_eval(
        base_model=dl.outputs["base_model"],
        val_data=prep.outputs["val_data"],
        config_yaml=config_yaml,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    baseline.set_gpu_limit(1).set_memory_limit("32G")
    k8s_ext.use_secret_as_env(baseline, secret_name=_SECRET, secret_key_to_env=_hf_map)

    # ── fine tune ──────────────────────────────────────────────────────────────
    ft = fine_tune(
        base_model=dl.outputs["base_model"],
        train_data=prep.outputs["train_data"],
        val_data=prep.outputs["val_data"],
        config_yaml=config_yaml,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    ft.after(baseline)
    ft.set_gpu_limit(1).set_memory_limit("64G")
    k8s_ext.use_secret_as_env(ft, secret_name=_SECRET, secret_key_to_env=_hf_map)

    # ── post fine-tune eval ────────────────────────────────────────────────────
    postft = post_finetune_eval(
        ft_model=ft.outputs["ft_model"],
        test_data=prep.outputs["test_data"],
        config_yaml=config_yaml,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    postft.set_gpu_limit(1).set_memory_limit("32G")
    k8s_ext.use_secret_as_env(postft, secret_name=_SECRET, secret_key_to_env=_hf_map)

    # ── deployment gate ────────────────────────────────────────────────────
    gate = deployment_gate(
        baseline_accuracy=baseline.outputs["baseline_accuracy"],
        baseline_auc=baseline.outputs["baseline_auc"],
        postft_accuracy=postft.outputs["postft_accuracy"],
        postft_auc=postft.outputs["postft_auc"],
        config_yaml=config_yaml,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )


***

## Build → `pipeline.py`

Run this cell after editing any `kfp_step` or `kfp_pipeline` cell or after changing
`processors.py` or `utils.py`. It regenerates the standalone `pipeline.py` file.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "scripts/build_pipeline.py"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

***

## Compile & Submit

In [ ]:
# Compile check — must pass before deploying
from kfp import compiler
from pipeline import pipeline
compiler.Compiler().compile(pipeline, "/tmp/pipeline_compiled.yaml")
print("Compile check: OK")

In [ ]:
import kfp, os
host = os.environ.get("KFP_HOST", "http://localhost:8890")
client = kfp.Client(host=host)
print(f"Connected to KFP at {host}")

In [ ]:
# Deploy via script (recommended — handles pipeline registration + MLflow experiment setup)
import subprocess, sys
result = subprocess.run(
    [sys.executable, "scripts/deploy_pipeline.py"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)